# Azure AI Search Integrated Vectorisation Notebook

This notebook is a runnable companion to the Bicep project. It shows how to use Azure AI Search indexers and skillsets to ingest documents, crack content, chunk text, generate embeddings and project each chunk into a vector index.

It is designed for workshop use:

1. Install dependencies.
2. Load environment variables.
3. Upload mixed document types to Azure Blob Storage.
4. Create an Azure AI Search index.
5. Create a data source connection.
6. Create a skillset with text splitting and Azure OpenAI embeddings.
7. Create and run an indexer.
8. Query with keyword, vector and hybrid search.
9. Analyse chunk sizes.

> Note: the SDK surface for integrated vectorisation has used preview classes. The cells include fallback imports and comments so developers can adapt if SDK names change.


In [ ]:
# Cell 1 - Install dependencies
# Run this once in a fresh notebook environment.
%pip install --upgrade --quiet azure-search-documents azure-identity azure-storage-blob python-dotenv tiktoken matplotlib numpy python-docx pillow


In [ ]:
# Cell 2 - Imports
import os
import math
from pathlib import Path
from typing import Iterable

import numpy as np
import matplotlib.pyplot as plt
import tiktoken
from dotenv import load_dotenv

from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient, ContentSettings

from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient, SearchIndexerClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchIndexer,
    SearchIndexerDataSourceConnection,
    SearchIndexerDataContainer,
    SearchField,
    SearchFieldDataType,
    VectorSearch,
    VectorSearchProfile,
    HnswVectorSearchAlgorithmConfiguration,
    AzureOpenAIEmbeddingSkill,
    SplitSkill,
)

# Preview / generated models used by integrated vectorisation samples.
# These imports can move between SDK versions, so keep them isolated here.
try:
    from azure.search.documents.indexes._generated.models import (
        SearchIndexerSkillset,
        AzureOpenAIVectorizer,
        AzureOpenAIParameters,
        SearchIndexerIndexProjections,
        SearchIndexerIndexProjectionSelector,
        SearchIndexerIndexProjectionsParameters,
        InputFieldMappingEntry,
        OutputFieldMappingEntry,
    )
except Exception as exc:
    raise ImportError(
        "Preview integrated vectorisation models were not found. "
        "Install/upgrade azure-search-documents or align the notebook with your SDK version."
    ) from exc


In [ ]:
# Cell 3 - Environment configuration
# Copy sample-ingestion/.env.example to sample-ingestion/.env and add values.

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
ENV_PATH = PROJECT_ROOT / "sample-ingestion" / ".env"
load_dotenv(ENV_PATH)

SEARCH_ENDPOINT = os.environ["SEARCH_ENDPOINT"].rstrip("/")
SEARCH_API_KEY = os.getenv("SEARCH_API_KEY", "")
SEARCH_INDEX_NAME = os.getenv("SEARCH_INDEX_NAME", "rag-documents-index-integrated")

AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY", "")

STORAGE_CONNECTION_STRING = os.environ["STORAGE_CONNECTION_STRING"]
BLOB_CONTAINER_NAME = os.getenv("BLOB_CONTAINER_NAME", "ai-search-raw-documents")

# Optional OCR settings. OCR needs an Azure AI services resource when you add OCR skills.
USE_OCR = os.getenv("USE_OCR", "false").lower() == "true"
AZURE_AI_SERVICES_ENDPOINT = os.getenv("AZURE_AI_SERVICES_ENDPOINT", "")
AZURE_AI_SERVICES_KEY = os.getenv("AZURE_AI_SERVICES_KEY", "")

credential = AzureKeyCredential(SEARCH_API_KEY) if SEARCH_API_KEY else DefaultAzureCredential()
index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)
indexer_client = SearchIndexerClient(endpoint=SEARCH_ENDPOINT, credential=credential)
search_client = SearchClient(endpoint=SEARCH_ENDPOINT, index_name=SEARCH_INDEX_NAME, credential=credential)

print("Configured index:", SEARCH_INDEX_NAME)
print("OCR enabled:", USE_OCR)


## Cell 4 - Upload mixed file types to Blob Storage

This is the block that demonstrates loading different document formats for indexer-based cracking:

- PDF: `.pdf`
- Word: `.docx`
- PowerPoint: `.pptx`
- OCR/image content: `.png`, `.jpg`, `.jpeg`, `.tif`, `.tiff`

The Azure AI Search indexer reads the blobs and the skillset performs chunking and vectorisation. For image-heavy documents or scanned files, add OCR/layout skills and set `USE_OCR=true` with an Azure AI services resource.


In [ ]:
# Cell 4 - Upload PDF, Word, PowerPoint and OCR/image files to Blob Storage

RAW_DOCS_DIR = PROJECT_ROOT / "sample-ingestion" / "data" / "raw-documents"
SUPPORTED_EXTENSIONS = {
    ".pdf": "application/pdf",
    ".docx": "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
    ".pptx": "application/vnd.openxmlformats-officedocument.presentationml.presentation",
    ".png": "image/png",
    ".jpg": "image/jpeg",
    ".jpeg": "image/jpeg",
    ".tif": "image/tiff",
    ".tiff": "image/tiff",
}

blob_service = BlobServiceClient.from_connection_string(STORAGE_CONNECTION_STRING)
container_client = blob_service.get_container_client(BLOB_CONTAINER_NAME)
try:
    container_client.create_container()
    print(f"Created container: {BLOB_CONTAINER_NAME}")
except Exception:
    print(f"Using existing container: {BLOB_CONTAINER_NAME}")

uploaded = []
for path in RAW_DOCS_DIR.rglob("*"):
    if not path.is_file():
        continue
    content_type = SUPPORTED_EXTENSIONS.get(path.suffix.lower())
    if not content_type:
        continue
    blob_name = path.relative_to(RAW_DOCS_DIR).as_posix()
    with path.open("rb") as handle:
        container_client.upload_blob(
            name=blob_name,
            data=handle,
            overwrite=True,
            content_settings=ContentSettings(content_type=content_type),
        )
    uploaded.append(blob_name)

print(f"Uploaded {len(uploaded)} files")
for name in uploaded:
    print(" -", name)


In [ ]:
# Cell 5 - Create the search index

def create_search_index(
    index_name: str,
    azure_openai_endpoint: str,
    azure_openai_embedding_deployment_id: str,
    azure_openai_key: str | None = None,
    dimensions: int = 1536,
) -> SearchIndex:
    fields = [
        SearchField(name="chunk_id", type=SearchFieldDataType.String, key=True, hidden=False, filterable=True),
        SearchField(name="parent_id", type=SearchFieldDataType.String, hidden=False, filterable=True),
        SearchField(name="chunk", type=SearchFieldDataType.String, hidden=False, searchable=True),
        SearchField(name="title", type=SearchFieldDataType.String, hidden=False, searchable=True, filterable=False, sortable=False, facetable=False),
        SearchField(name="source_file", type=SearchFieldDataType.String, hidden=False, searchable=True, filterable=True, sortable=True, facetable=True),
        SearchField(name="content_type", type=SearchFieldDataType.String, hidden=False, searchable=False, filterable=True, sortable=True, facetable=True),
        SearchField(name="language", type=SearchFieldDataType.String, hidden=False, searchable=False, filterable=True, sortable=True, facetable=True),
        SearchField(name="classification", type=SearchFieldDataType.String, hidden=False, searchable=False, filterable=True, sortable=True, facetable=True),
        SearchField(name="group_ids", type=SearchFieldDataType.Collection(SearchFieldDataType.String), hidden=False, searchable=False, filterable=True, facetable=True),
        SearchField(
            name="vector",
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            hidden=False,
            searchable=True,
            filterable=False,
            sortable=False,
            facetable=False,
            vector_search_dimensions=dimensions,
            vector_search_profile="profile",
        ),
    ]

    vector_search = VectorSearch(
        profiles=[
            VectorSearchProfile(
                name="profile",
                algorithm="hnsw-algorithm",
                vectorizer="azure-openai-vectorizer",
            )
        ],
        algorithms=[
            HnswVectorSearchAlgorithmConfiguration(name="hnsw-algorithm")
        ],
        vectorizers=[
            AzureOpenAIVectorizer(
                name="azure-openai-vectorizer",
                parameters=AzureOpenAIParameters(
                    resource_uri=azure_openai_endpoint,
                    deployment_id=azure_openai_embedding_deployment_id,
                    api_key=azure_openai_key,
                ),
            )
        ],
    )

    return SearchIndex(name=index_name, fields=fields, vector_search=vector_search)

index = create_search_index(
    SEARCH_INDEX_NAME,
    AZURE_OPENAI_ENDPOINT,
    AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
    AZURE_OPENAI_API_KEY or None,
)
index_client.create_or_update_index(index)
print("Created or updated index:", SEARCH_INDEX_NAME)


In [ ]:
# Cell 6 - Create data source connection for Azure Blob Storage

def create_search_datasource(
    datasource_name: str,
    connection_string: str,
    container_name: str,
) -> SearchIndexerDataSourceConnection:
    return SearchIndexerDataSourceConnection(
        name=datasource_name,
        type="azureblob",
        connection_string=connection_string,
        container=SearchIndexerDataContainer(name=container_name),
    )

DATA_SOURCE_NAME = f"{SEARCH_INDEX_NAME}-blob-datasource"
datasource = create_search_datasource(DATA_SOURCE_NAME, STORAGE_CONNECTION_STRING, BLOB_CONTAINER_NAME)
indexer_client.create_or_update_data_source_connection(datasource)
print("Created or updated data source:", DATA_SOURCE_NAME)


In [ ]:
# Cell 7 - Create skillset with chunking and embeddings

def create_search_skillset(
    skillset_name: str,
    index_name: str,
    azure_openai_endpoint: str,
    azure_openai_embedding_deployment_id: str,
    azure_openai_key: str | None = None,
    text_split_mode: str = "pages",
    maximum_page_length: int = 2000,
    page_overlap_length: int = 500,
) -> SearchIndexerSkillset:
    split_skill = SplitSkill(
        name="Text Splitter",
        description="Split extracted document text into overlapping chunks",
        default_language_code="en",
        text_split_mode=text_split_mode,
        maximum_page_length=maximum_page_length,
        page_overlap_length=page_overlap_length,
        context="/document",
        inputs=[InputFieldMappingEntry(name="text", source="/document/content")],
        outputs=[OutputFieldMappingEntry(name="textItems", target_name="pages")],
    )

    embedding_skill = AzureOpenAIEmbeddingSkill(
        name="Embeddings",
        description="Generate embeddings for each chunk",
        resource_uri=azure_openai_endpoint,
        deployment_id=azure_openai_embedding_deployment_id,
        api_key=azure_openai_key,
        context="/document/pages/*",
        inputs=[InputFieldMappingEntry(name="text", source="/document/pages/*")],
        outputs=[OutputFieldMappingEntry(name="embedding", target_name="vector")],
    )

    projections = SearchIndexerIndexProjections(
        selectors=[
            SearchIndexerIndexProjectionSelector(
                target_index_name=index_name,
                parent_key_field_name="parent_id",
                source_context="/document/pages/*",
                mappings=[
                    InputFieldMappingEntry(name="chunk", source="/document/pages/*"),
                    InputFieldMappingEntry(name="vector", source="/document/pages/*/vector"),
                    InputFieldMappingEntry(name="title", source="/document/metadata_storage_name"),
                    InputFieldMappingEntry(name="source_file", source="/document/metadata_storage_name"),
                    InputFieldMappingEntry(name="content_type", source="/document/metadata_content_type"),
                ],
            )
        ],
        parameters=SearchIndexerIndexProjectionsParameters(
            projection_mode="skipIndexingParentDocuments"
        ),
    )

    return SearchIndexerSkillset(
        name=skillset_name,
        description="Crack documents, split content into chunks and generate embeddings",
        skills=[split_skill, embedding_skill],
        index_projections=projections,
    )

SKILLSET_NAME = f"{SEARCH_INDEX_NAME}-skillset"
skillset = create_search_skillset(
    SKILLSET_NAME,
    SEARCH_INDEX_NAME,
    AZURE_OPENAI_ENDPOINT,
    AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
    AZURE_OPENAI_API_KEY or None,
)
indexer_client.create_or_update_skillset(skillset)
print("Created or updated skillset:", SKILLSET_NAME)


In [ ]:
# Cell 8 - Create and run the indexer

def create_search_indexer(
    indexer_name: str,
    skillset_name: str,
    datasource_name: str,
    index_name: str,
) -> SearchIndexer:
    return SearchIndexer(
        name=indexer_name,
        data_source_name=datasource_name,
        target_index_name=index_name,
        skillset_name=skillset_name,
    )

INDEXER_NAME = f"{SEARCH_INDEX_NAME}-indexer"
indexer = create_search_indexer(INDEXER_NAME, SKILLSET_NAME, DATA_SOURCE_NAME, SEARCH_INDEX_NAME)
indexer_client.create_or_update_indexer(indexer)
indexer_client.run_indexer(INDEXER_NAME)
print("Started indexer:", INDEXER_NAME)


In [ ]:
# Cell 9 - Check indexer status
status = indexer_client.get_indexer_status(INDEXER_NAME)
print("Status:", status.status)
print("Last result:", status.last_result.status if status.last_result else None)
if status.last_result and status.last_result.errors:
    for error in status.last_result.errors[:10]:
        print("ERROR:", error)
if status.last_result and status.last_result.warnings:
    for warning in status.last_result.warnings[:10]:
        print("WARNING:", warning)


In [ ]:
# Cell 10 - Keyword search
results = search_client.search(
    search_text="private endpoint",
    select=["chunk_id", "title", "source_file", "chunk"],
    top=5,
)
for result in results:
    print("---")
    print(result["title"], result.get("source_file"))
    print(result["chunk"][:500])


In [ ]:
# Cell 11 - Vector query using the integrated vectorizer
from azure.search.documents.models import VectorizableTextQuery

vector_query = VectorizableTextQuery(
    text="How do I secure Azure AI Search with private endpoints?",
    k_nearest_neighbors=5,
    fields="vector",
)

results = search_client.search(
    search_text=None,
    vector_queries=[vector_query],
    select=["chunk_id", "title", "source_file", "chunk"],
    top=5,
)
for result in results:
    print("---", result.get("@search.score"))
    print(result["title"], result.get("source_file"))
    print(result["chunk"][:500])


In [ ]:
# Cell 12 - Hybrid search: keyword + vector
query = "private endpoint DNS for Azure AI Search"
vector_query = VectorizableTextQuery(
    text=query,
    k_nearest_neighbors=20,
    fields="vector",
)

results = search_client.search(
    search_text=query,
    vector_queries=[vector_query],
    select=["chunk_id", "title", "source_file", "chunk"],
    top=5,
)
for result in results:
    print("---", result.get("@search.score"))
    print(result["title"], result.get("source_file"))
    print(result["chunk"][:500])


In [ ]:
# Cell 13 - Optional security trimming filter example
# This works after your ingestion pipeline populates group_ids or classification fields.

caller_groups = ["ai-platform", "security"]
groups_csv = ",".join(caller_groups)
security_filter = f"group_ids/any(g: search.in(g, '{groups_csv}')) and classification ne 'HighlyConfidential'"

# Uncomment after you map group_ids/classification in your skillset or ingestion code.
# results = search_client.search(
#     search_text=query,
#     vector_queries=[vector_query],
#     filter=security_filter,
#     select=["chunk_id", "title", "source_file", "classification", "chunk"],
#     top=5,
# )


In [ ]:
# Cell 14 - Retrieve chunks and analyse token length distribution

def get_chunks(search_client: SearchClient, top: int = 100000) -> list[str]:
    results = search_client.search(
        search_text="*",
        top=top,
        select=["chunk_id", "chunk"],
    )
    chunks = {}
    for result in results:
        chunk_id = result["chunk_id"]
        chunks[chunk_id] = result["chunk"]
    return [chunks[key] for key in sorted(chunks.keys())]


def get_encoding_for_model(model_name: str = "gpt-4o-mini"):
    try:
        return tiktoken.encoding_for_model(model_name)
    except KeyError:
        return tiktoken.get_encoding("cl100k_base")


def get_token_length(text: str, model_name: str = "gpt-4o-mini") -> int:
    encoding = get_encoding_for_model(model_name)
    return len(encoding.encode(text or ""))


def round_to_lowest_multiple(number: int, multiple: int) -> int:
    return math.floor(number / multiple) * multiple


def round_to_highest_multiple(number: int, multiple: int) -> int:
    return math.ceil(number / multiple) * multiple


def plot_chunk_histogram(
    chunks: list[str],
    length_fn=get_token_length,
    title: str = "Chunk token length distribution",
    xlabel: str = "Tokens per chunk",
    ylabel: str = "Chunk count",
):
    ys = [length_fn(chunk) for chunk in chunks]
    if not ys:
        print("No chunks to plot")
        return
    min_y = min(ys)
    max_y = max(ys)
    n, bins, patches = plt.hist(ys, bins=25)
    max_freq = max(n) if len(n) else 1
    tick_step = max(int(round_to_lowest_multiple(max_y - min_y, 100) / 5), 100)
    max_xtick = round_to_highest_multiple(max_y, tick_step)
    min_xtick = round_to_lowest_multiple(min_y, tick_step)
    xticks = list(np.arange(start=min_xtick, stop=max_xtick + tick_step, step=tick_step))
    plt.xticks(xticks)
    plt.xlim(min_xtick, max_xtick)
    plt.ylim(0, max_freq + 1)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.show()

chunks = get_chunks(search_client)
print("Chunk count:", len(chunks))
plot_chunk_histogram(chunks)


In [ ]:
# Cell 15 - Tuning notes
print("""
Recommended tuning loop:
1. Inspect chunk histogram.
2. If chunks are too large, reduce maximum_page_length or switch to token/semantic chunking.
3. If chunks are too small, increase chunk length or reduce overlap.
4. If answers miss context across boundaries, increase overlap.
5. If retrieval is noisy, improve metadata, use filters, tune semantic config, or split by headings.
6. For production, add RBAC/sensitivity fields before the content is exposed to an agent.
""")
